In [45]:
import pandas as pd 
import numpy as np
import time
from sklearn.linear_model import LinearRegression,Ridge,Lasso,ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)



In [46]:
X_train = pd.DataFrame(pd.read_csv("../data/preprocess/X_train.csv",index_col=0))
X_test = pd.DataFrame(pd.read_csv("../data/preprocess/X_test.csv",index_col=0))
y_train = pd.DataFrame(pd.read_csv("../data/preprocess/y_train.csv"))
y_test = pd.DataFrame(pd.read_csv("../data/preprocess/y_test.csv"))

X_train

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender
5810,27,79,Low,High,Yes,8,63,High,Yes,2,Low,Medium,Public,Negative,5,No,College,Moderate,Female
1268,16,86,High,Medium,Yes,7,94,Medium,Yes,2,Low,High,Public,Neutral,3,No,High School,Moderate,Female
414,22,87,Low,Medium,No,8,83,Low,Yes,1,Low,Medium,Public,Neutral,1,No,College,Far,Male
4745,18,100,High,Medium,Yes,10,86,Medium,Yes,1,Medium,Medium,Public,Neutral,3,No,High School,Near,Male
654,35,78,High,Low,Yes,10,99,Medium,Yes,1,Low,Medium,Private,Positive,2,No,High School,Near,Male
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3772,15,82,Medium,Medium,Yes,7,93,Medium,Yes,3,Low,High,Public,Negative,2,No,High School,Moderate,Female
5191,20,65,Medium,Medium,Yes,8,97,High,Yes,0,Low,Medium,Public,Negative,3,No,College,Near,Female
5226,17,64,High,Low,Yes,10,63,Medium,Yes,0,High,Medium,Public,Positive,3,No,High School,Moderate,Female
5390,16,100,High,High,Yes,7,82,Medium,Yes,2,High,Medium,Public,Positive,2,No,High School,Near,Male


In [47]:
def encode_features(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()

    mappings = {
        "Parental_Involvement": {
            "Low": 0,
            "Medium": 1,
            "High": 2,
        },

        "Access_to_Resources": {
            "Low": 0,
            "Medium": 1,
            "High": 2,
        },

        "Motivation_Level": {
            "Low": 0,
            "Medium": 1,
            "High": 2,
        },

        "Family_Income": {
            "Low": 0,
            "Medium": 1,
            "High": 2,
        },

        "Teacher_Quality": {
            "Low": 0,
            "Medium": 1,
            "High": 2,
        },

        "Peer_Influence": {
            "Negative": 0,
            "Neutral": 1,
            "Positive": 2,
        },

        "Parental_Education_Level": {
            "High School": 0,
            "College": 1,
            "Postgraduate": 2,
        },

        "Distance_from_Home": {
            "Near": 0,
            "Moderate": 1,
            "Far": 2,
        },

        "Extracurricular_Activities": {
            "No": 0,
            "Yes": 1,
        },

        "Internet_Access": {
            "No": 0,
            "Yes": 1,
        },

        "Learning_Disabilities": {
            "No": 0,
            "Yes": 1,
        },

        "Gender": {
            "Female": 0,
            "Male": 1,
        },
    }

    for column, mapping in mappings.items():
        df[column] = df[column].map(mapping)

    df = pd.get_dummies(
        df,
        columns=["School_Type"],
        drop_first=True,
        dtype=int
    )

    return df

In [48]:
X_train = encode_features(X_train)
X_test = encode_features(X_test)
X_test

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,School_Type_Public
743,20,71,1,0,0,7,87,2,1,1,1,1,0,5,0,0,0,1,1
5551,22,71,1,0,1,7,98,0,1,2,0,2,1,2,0,0,1,0,1
3442,21,91,2,1,1,6,53,2,1,1,1,1,2,3,0,2,0,0,1
6571,12,91,1,0,1,8,81,0,1,0,0,0,2,4,0,0,1,1,1
4204,21,63,0,2,1,8,95,1,1,2,2,1,1,5,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4709,20,78,2,1,1,10,85,1,1,1,2,2,0,3,0,0,0,1,0
3664,27,90,2,2,0,7,92,1,1,0,0,1,1,3,0,0,0,0,0
5231,23,72,0,1,0,6,61,2,1,1,2,1,2,5,0,0,0,1,1
1773,21,76,0,1,0,6,50,0,1,2,0,1,0,4,0,2,1,0,1


In [49]:
models = {
    "Linear Regression":LinearRegression(),
    "Ridge":Ridge(random_state=42),
    "Lasso":Lasso(random_state=42),
    "ElasticNet":ElasticNet(random_state=42),
    "DecisionTree Regressor": DecisionTreeRegressor(random_state=42),
    "RandomForest Regressor": RandomForestRegressor(random_state=42),
    "Gradient Boosting Regressor":GradientBoostingRegressor(random_state=42),
    "extra tree":ExtraTreesRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42) 
}

In [50]:
result = []

for name,model in models.items():
    start = time.time()
    model.fit(X_train,y_train)

    y_pred = model.predict(X_test)
    end = time.time()
    print(f"{name} took {end-start:.2f} seconds")
    mae = mean_absolute_error(y_test,y_pred)
    mse = mean_squared_error(y_test,y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test,y_pred)

    result.append({
        "Model": name,
        "mae":mae,
        "mse":mse,
        "rmse":rmse,
        "r2 score": r2
    })
    

Linear Regression took 0.01 seconds
Ridge took 0.01 seconds
Lasso took 0.01 seconds
ElasticNet took 0.01 seconds
DecisionTree Regressor took 0.06 seconds


c:\Users\dell\Desktop\AI-ML-Internship-Codomax\venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForest Regressor took 2.33 seconds


c:\Users\dell\Desktop\AI-ML-Internship-Codomax\venv\Lib\site-packages\sklearn\ensemble\_gb.py:691: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)  # TODO: Is this still required?


Gradient Boosting Regressor took 0.59 seconds


c:\Users\dell\Desktop\AI-ML-Internship-Codomax\venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


extra tree took 2.30 seconds


c:\Users\dell\Desktop\AI-ML-Internship-Codomax\venv\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


AdaBoost took 0.33 seconds


In [51]:
results_df = pd.DataFrame(result)
results_df = results_df.sort_values(
    by="r2 score",
    ascending=False
).reset_index(drop=True)

results_df

,Model,mae,mse,rmse,r2 score
0,Ridge,0.444268,3.237815,1.799393,0.770937
1,Linear Regression,0.444289,3.237946,1.799429,0.770928
2,Gradient Boosting Regressor,0.788342,3.755416,1.937889,0.734319
3,extra tree,0.983215,4.314291,2.077087,0.694781
4,RandomForest Regressor,1.081755,4.707649,2.169712,0.666953
5,ElasticNet,1.323595,5.218626,2.284431,0.630803
6,Lasso,1.374618,5.400586,2.323916,0.617930
7,DecisionTree Regressor,1.576399,8.611195,2.934484,0.390792
8,AdaBoost,4.959683,34.128548,5.841964,-1.414459
